In [1]:
import numpy as np 
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem.porter import PorterStemmer
nltk.download('stopwords')
nltk.download('ktinker')
import re 
import string
import os
path= '../data/questions.csv'
df=pd.read_csv(path)

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/avakash/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Error loading ktinker: Package 'ktinker' not found in
[nltk_data]     index


In [2]:
df=df.reset_index(drop=True)

In [3]:
device='cuda'

In [4]:
df.head()

,id,qid1,qid2,question1,question2,is_duplicate
0,0,1,2,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,0
1,1,3,4,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,0
2,2,5,6,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,0
3,3,7,8,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,0
4,4,9,10,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,0


In [5]:
# df=df.drop(columns=['id','qid1','qid2'])
df=df.dropna()
df.info()


<class 'pandas.core.frame.DataFrame'>
Index: 404348 entries, 0 to 404350
Data columns (total 6 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   id            404348 non-null  int64 
 1   qid1          404348 non-null  int64 
 2   qid2          404348 non-null  int64 
 3   question1     404348 non-null  object
 4   question2     404348 non-null  object
 5   is_duplicate  404348 non-null  int64 
dtypes: int64(4), object(2)
memory usage: 21.6+ MB


In [6]:
def icon1(l):
    p=str(l).replace(",",' ,') 
    p=p.replace("?"," ?")
    p=p.replace("/"," ") 
    p=p.replace("."," . ")
    p=p.replace('-',' - ')
    p=p.replace("("," ( ") 
    p=p.replace(")"," ) ") 
    p=p.replace("<"," < ") 
    p=p.replace(">"," > ") 
    p=p.replace("\""," \" ") 
    p=p.replace("\'"," \' ")
    p=re.sub(r'[^a-zA-Z\s]','',p)
    p=p.lower()
    return p.split()

In [7]:
df['transformation1']=df['question1'].apply(icon1)
df['transformation2']=df['question2'].apply(icon1)

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 404348 entries, 0 to 404350
Data columns (total 8 columns):
 #   Column           Non-Null Count   Dtype 
---  ------           --------------   ----- 
 0   id               404348 non-null  int64 
 1   qid1             404348 non-null  int64 
 2   qid2             404348 non-null  int64 
 3   question1        404348 non-null  object
 4   question2        404348 non-null  object
 5   is_duplicate     404348 non-null  int64 
 6   transformation1  404348 non-null  object
 7   transformation2  404348 non-null  object
dtypes: int64(4), object(4)
memory usage: 27.8+ MB


In [9]:
vocab=set()
for i in df.question1:
    p=icon1(str(i))
    for j in p:
        vocab.add(j)
for i in df.question2:
    p=icon1(str(i))
    for j in p:
        vocab.add(j)
vocab=list(vocab)

In [10]:
len(vocab)

80808

In [11]:
vocab[:10]

['extraterrestrial',
 'drowns',
 'terminology',
 'tweeted',
 'biographical',
 'bonaparte',
 'coopers',
 'pues',
 'unjam',
 'maciel']

In [83]:
id2voc={}
voc2id={}
idx=0
for i in vocab:
    id2voc[idx]=i
    voc2id[i]=idx
    idx+=1
id2voc[idx]='<UNK>'
voc2id['<UNK>']=idx


In [13]:
len(voc2id)

80809

In [14]:
import torch
import torch.nn as nn
vocab_size=len(voc2id)
embedding_dim=728

In [15]:
E=nn.Embedding(vocab_size,embedding_dim).to(device)

In [16]:
sample=df.question1[0]
sample=icon1(sample)
sample_test=[voc2id[i] for i in sample]
sample_test

[51729,
 56289,
 5559,
 14592,
 29338,
 14592,
 31348,
 46946,
 40382,
 27002,
 38041,
 72952,
 27002,
 58198]

In [17]:
x=torch.tensor(sample_test)
embed=E(x.to(device))
print(embed.shape)

torch.Size([14, 728])


In [18]:
avg=embed.mean(dim=0)
avg.shape

torch.Size([728])

In [61]:
def floss(i1,i2):
    return nn.functional.cosine_similarity(i1,i2)

class Embeds(nn.Module):
    def __init__(self,voc2id,E):
        super().__init__()
        self.voc2id=voc2id
        self.E=E
    def forward(self,x):
        emb=self.E(x)
        mask=(x!=80808).unsqueeze(-1).float().to(device)
        emb=emb*mask
        summed=emb.sum(dim=1)
        lengths=mask.sum(dim=1).clamp(min=1)
        pooled=summed/lengths
        return pooled

In [62]:
model=Embeds(voc2id=voc2id,E=E).to(device)


In [63]:
df.loc[df['is_duplicate']==0,'is_duplicate']=-1

In [298]:
import torch.optim as optim
lossfn=nn.CosineEmbeddingLoss()
opt=optim.Adam(model.parameters(),lr=5e-4)
model.E.weight.requires_grad

True

In [41]:
def valid_sentence(sentence):
    return len(sentence) > 0

In [ ]:
pairs=[]
removed=0
for i in range(len(df)):
    q1=df.transformation1.iloc[i]
    q2=df.transformation2.iloc[i]
    label=df.is_duplicate.iloc[i]
    if valid_sentence(q1) and valid_sentence(q2):
        pairs.append((q1,q2,label))
    else:
        print(removed)
        removed+=1
print("Removed:",removed)
print("Remaining:",len(pairs))

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
Removed: 28
Remaining: 404320


In [43]:
import torch
import pickle
torch.save(model.state_dict(), "embed_model.pth")
with open("voc2id.pkl", "wb") as f:
    pickle.dump(voc2id,f)
with open("id2voc.pkl","wb") as f:
    pickle.dump(id2voc,f)

In [44]:
#batch upgrade
from torch.utils.data import Dataset,DataLoader
class pairDataset(Dataset):
    def __init__(self,pairs):
        self.pairs=pairs
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self,idx):
        s1,s2,label=self.pairs[idx]
        return s1,s2,label

In [45]:

with open("voc2id.pkl", "rb") as f:
    voc2id=pickle.load(f)
    print(voc2id['<UNK>'])

80808


In [187]:
from torch.nn.utils.rnn import pad_sequence
import pickle
def collate_fn(batch):
    s1b=[]
    s2b=[]
    label=[]
    with open("voc2id.pkl", "rb") as f:
        voc2id=pickle.load(f)

    for s1,s2,labels in batch:
        ids1=torch.tensor([voc2id.get(i,80808) for i in s1],dtype=torch.long)
        ids2=torch.tensor([voc2id.get(i,80808) for i in s2],dtype=torch.long)
        s1b.append(ids1)
        s2b.append(ids2)
        label.append(labels)
    s1b=pad_sequence(s1b,batch_first=True,padding_value=voc2id["<UNK>"])
    s2b=pad_sequence(s2b,batch_first=True,padding_value=voc2id["<UNK>"])
    label=torch.tensor(label).float()
    return s1b,s2b,label

In [188]:
from sklearn.model_selection import train_test_split
ds=pairDataset(pairs)
train,test=train_test_split(ds,test_size=.2,random_state=2)

loader=DataLoader(train,batch_size=256,shuffle=True,collate_fn=collate_fn)
tel=DataLoader(test,batch_size=256,shuffle=True,collate_fn=collate_fn)

In [106]:
x,y,l=next(iter(tel))

In [107]:
target=next(iter(tel))[2]
target

tensor([-1., -1., -1., -1., -1.,  1., -1., -1., -1.,  1.,  1., -1.,  1., -1.,
        -1., -1., -1., -1., -1., -1.,  1., -1., -1., -1.,  1., -1.,  1.,  1.,
        -1., -1.,  1., -1., -1., -1., -1., -1., -1.,  1.,  1.,  1.,  1.,  1.,
         1.,  1., -1., -1., -1.,  1., -1.,  1., -1.,  1.,  1., -1.,  1., -1.,
         1., -1., -1., -1., -1., -1., -1.,  1., -1., -1.,  1.,  1., -1., -1.,
        -1.,  1., -1., -1., -1., -1., -1.,  1., -1., -1., -1., -1., -1., -1.,
         1., -1., -1.,  1., -1.,  1., -1., -1., -1., -1., -1., -1., -1., -1.,
        -1., -1., -1., -1., -1.,  1., -1.,  1., -1., -1., -1., -1.,  1., -1.,
         1., -1., -1., -1.,  1., -1.,  1.,  1.,  1., -1., -1.,  1., -1., -1.,
        -1., -1., -1., -1.,  1., -1., -1., -1., -1., -1., -1.,  1., -1.,  1.,
        -1.,  1., -1., -1.,  1., -1., -1., -1., -1.,  1.,  1., -1., -1., -1.,
        -1.,  1., -1., -1., -1.,  1.,  1., -1.,  1.,  1., -1., -1., -1., -1.,
         1., -1., -1.,  1., -1., -1.,  1., -1., -1.,  1., -1., -

In [308]:
epoch=6
model.train()
for e in range(epoch):
    total_loss=0
    for s1,s2,label in loader:
        opt.zero_grad()
        s1,s2=s1.to(device),s2.to(device)
        e1=model(s1)
        e2=model(s2)
        target=label.to(device)
        loss=lossfn(e1,e2,target)
        loss.backward()
        opt.step()
        total_loss+=loss.item()
    print(f'epoch {e} : {total_loss}')

epoch 0 : 276.85328693687916
epoch 1 : 274.4927167594433
epoch 2 : 272.19369065761566
epoch 3 : 269.98439821600914
epoch 4 : 267.81343038380146
epoch 5 : 265.7183346748352


In [313]:
torch.save(model,'final.pth')

In [296]:
model=torch.load('final.pth',weights_only=False)

In [311]:
model.eval()
x,y,l=next(iter(tel))
e1=model(x.to(device))
e2=model(y.to(device))
sim=e1[0].dot(e2[0])/(torch.norm(e1[0])*torch.norm(e2[0]))
sim,l[0]

(tensor(-0.0160, device='cuda:0', grad_fn=<DivBackward0>), tensor(-1.))

In [312]:
[id2voc[int(i)] for i in x[0] if id2voc[int(i)]!="<UNK>"],[id2voc[int(i)] for i in y[0] if id2voc[int(i)]!="<UNK>"]

(['what',
  'are',
  'the',
  'exact',
  'differences',
  'between',
  'the',
  'snapdragon',
  'and',
  'exynos',
  'chips',
  'on',
  'the',
  'samsung',
  'galaxy',
  's',
  'what',
  'would',
  'be',
  'better',
  'in',
  'multiple',
  'scenarios'],
 ['was', 'samsung', 'galaxy', 's', 'a', 'hit'])

In [300]:
x="i eat cat"
y="cat eat me"
x=icon1(x) 
y=icon1(y) 

In [301]:
x,y

(['i', 'eat', 'cat'], ['cat', 'eat', 'me'])

In [302]:
p=(x,y,-1)
data=pairDataset([p])

In [303]:
data[0]

(['i', 'eat', 'cat'], ['cat', 'eat', 'me'], -1)

In [304]:
te=DataLoader(data,batch_size=1,collate_fn=collate_fn)

In [309]:
model.eval()
x,y,l=next(iter(te))
e1=model(x.to(device))
e2=model(y.to(device))
sim=e1[0].dot(e2[0])/(torch.norm(e1[0])*torch.norm(e2[0]))
sim,l[0]

(tensor(0.7684, device='cuda:0', grad_fn=<DivBackward0>), tensor(-1.))

In [310]:
[id2voc[int(i)] for i in x[0] if id2voc[int(i)]!="<UNK>"],[id2voc[int(i)] for i in y[0] if id2voc[int(i)]!="<UNK>"]

(['i', 'eat', 'cat'], ['cat', 'eat', 'me'])